# 🚀 Google Colab GPU Worker — Multimodal RAG Financial

Kiến trúc Hybrid: **Web UI, SQLite/FTS5 và bộ tính toán `Decimal` chạy trên máy local**, còn **Google Colab (T4 / L4 / A100)** chỉ chạy GPU API cho:
- VLM (`Qwen2.5-VL-3B-Instruct` 4-bit) đọc biểu đồ ảnh và tổng hợp thuyết minh có trích dẫn
- Dense Embeddings (`BAAI/bge-m3`) trên CUDA
- Remote OCR trên GPU

> Chọn **Runtime → Change runtime type → T4 GPU** (hoặc L4/A100) trước khi chạy.

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_URL = "https://github.com/dongdx2k5/Financial-Multimodal-RAG.git"
REPO_DIR = "/content/Financial-Multimodal-RAG"
if not os.path.exists(REPO_DIR):
    !git clone --branch main {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

In [ ]:
!pip install -q fastapi uvicorn pillow accelerate bitsandbytes transformers sentence-transformers rapidocr onnxruntime-gpu
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import os
import re
import secrets
import subprocess
import time
import urllib.request

TOKEN = secrets.token_urlsafe(24)
env = {
    **os.environ,
    "COLAB_API_TOKEN": TOKEN,
    "COLAB_VLM_MODEL": "Qwen/Qwen2.5-VL-3B-Instruct",
    "COLAB_EMBED_MODEL": "BAAI/bge-m3",
    "COLAB_4BIT": "true",
}

with open("/tmp/colab_worker.log", "w", encoding="utf-8") as worker_log:
    worker_proc = subprocess.Popen(
        ["python", "-m", "uvicorn", "scripts.colab_gpu_server:app", "--host", "127.0.0.1", "--port", "8008"],
        cwd="/content/Financial-Multimodal-RAG",
        env=env,
        stdout=worker_log,
        stderr=subprocess.STDOUT,
    )

for _ in range(20):
    try:
        req = urllib.request.Request("http://127.0.0.1:8008/health", headers={"Authorization": f"Bearer {TOKEN}"})
        urllib.request.urlopen(req, timeout=2)
        break
    except OSError:
        time.sleep(1)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8008", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
if tunnel_proc.stdout is not None:
    for _ in range(40):
        line = tunnel_proc.stdout.readline()
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

print("=== COPY VÀO FILE .env TRÊN MÁY LOCAL ===")
print(f"COLAB_GPU_URL={public_url}")
print(f"COLAB_API_TOKEN={TOKEN}")